# HireFlow demo: multi-agent trace

Five agents plus a coordinator run as a **sequential pipeline** over one **shared candidate pool**. Scores and decisions are plain Python tools; a model is used only to polish one sentence, with a template fallback.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # project root

from hireflow import llm
from hireflow.agents.coordinator import PipelineCoordinator

providers = [p["name"] for p in llm.active_providers()]
print("Model providers configured:", providers or "none (template sentence will be used)")

Model providers configured: none (template sentence will be used)


## 1. Run the pipeline

In [2]:
coordinator = PipelineCoordinator()
pool = coordinator.run()
print(f"Run id {pool.log.run_id}: {len(pool.all())} resumes processed")
print("Stages in order:", " -> ".join(name for name, _ in coordinator.stages))

Run id 9bdc9c96: 8 resumes processed
Stages in order: parse -> match -> rank -> bias_check -> schedule


## 2. Multi-agent trace (who did what, in order)

Tool calls are summarised per agent; every individual call is in `logs/run_log.txt`.

In [3]:
from collections import Counter

calls = Counter(e["agent"] for e in pool.log.entries if e["event"] == "tool_call")
for e in pool.log.entries:
    if e["event"] != "tool_call":
        print(f"{e['ts'][11:]}  {e['agent']:<17} {e['event']:<19} {e['detail'][:110]}")
print()
print("Tool calls made through Semantic Kernel:")
for agent, n in calls.items():
    print(f"  {agent:<17} {n}")

22:27:27  system            run_started         
22:27:27  coordinator       inputs_loaded       job 'Backend Python Developer', 8 resumes
22:27:27  coordinator       stage_started       parse
22:27:27  parser_agent      parsed              resume_01 Priya Sharma: 5 skills, 4 yrs, B.Tech Computer Science
22:27:27  parser_agent      parsed              resume_02 Rahul Kumar: 4 skills, 3 yrs, B.Sc Information Technology
22:27:27  parser_agent      parsed              resume_03 Aman Tiwari: 3 skills, 5 yrs, M.Tech Software Engineering
22:27:27  parser_agent      parsed              resume_04 Sneha Iyer: 7 skills, 6 yrs, MCA
22:27:27  parser_agent      parsed              resume_05 Vikram Singh: 3 skills, 2 yrs, B.E. Computer Engineering
22:27:27  parser_agent      incomplete_profile  resume_06 Neha Gupta: no Skills section found - marked incomplete, nothing guessed
22:27:27  parser_agent      parsed              resume_07 Arjun Mehta: 4 skills, 0 yrs, B.Tech Computer Science
22:27:27  par

## 3. Final shortlist

In [4]:
print(pool.final_report)

Shortlist for: Backend Python Developer
----------------------------------------------------------------------------------------------------
1. Sneha Iyer | score 100 | Mon 17 Nov, 10:00
     Sneha Iyer: 100/100. Meets 4 of 4 required skills; 6 yrs experience (min 3); education MCA.
2. Priya Sharma | score 92 | Mon 17 Nov, 11:00
     Priya Sharma: 92/100. Meets 4 of 4 required skills; 4 yrs experience (min 3); education B.Tech Computer Science.
3. Kavya Nair | score 88.5 | Tue 18 Nov, 10:00
     Kavya Nair: 88.5/100. Meets 4 of 4 required skills; 3.5 yrs experience (min 3); education Diploma in Computer Applications.
4. Rahul Kumar | score 76 | NOT_SHORTLISTED
     Rahul Kumar: 76/100. Meets 3 of 4 required skills (missing: REST APIs); 3 yrs experience (min 3); education B.Sc Information Technology.
5. Aman Tiwari | score 64 | NOT_SHORTLISTED
     Aman Tiwari: 64/100. Meets 2 of 4 required skills (missing: SQL, REST APIs); 5 yrs experience (min 3); education M.Tech Software Engineering

## 4. Every point justified against a named requirement (top candidate)

In [5]:
top = min((c for c in pool.all() if c.rank), key=lambda c: c.rank)
print(f"{top.name}: {top.score:g}/100")
for row in top.breakdown:
    print(f"  {'met ' if row['met'] else 'MISS'}  {row['requirement']:<45} {row['points']:>5g} / {row['max']:g}")

Sneha Iyer: 100/100
  met   Required skill: Python                           12 / 12
  met   Required skill: SQL                              12 / 12
  met   Required skill: REST APIs                        12 / 12
  met   Required skill: Git                              12 / 12
  met   Preferred skill: Docker                           4 / 4
  met   Preferred skill: AWS                              4 / 4
  met   Preferred skill: FastAPI                          4 / 4
  met   Experience: 6 yrs vs minimum 3                   25 / 25
  met   Education: master vs minimum bachelor            15 / 15


## 5. Honest failure: a resume with no Skills section

In [6]:
neha = pool.candidates["resume_06"]
print("skills found :", neha.skills, "(nothing guessed)")
print("complete     :", neha.complete, "|", neha.issues)
print("rank / slot  :", neha.rank, "/", neha.slot, "| status:", neha.status)
print("rationale    :", neha.rationale)

skills found : [] (nothing guessed)
complete     : False | ['no Skills section found']
rank / slot  : None / None | status: needs_review
rationale    : Neha Gupta: incomplete profile (no Skills section found) - needs manual review, not ranked.


## 6. Bias Guard catches a leaking rationale

We simulate a model that wrongly mentions protected attributes.

In [7]:
import tempfile, pathlib
from hireflow import llm as llm_module

original = llm_module.generate_text
llm_module.generate_text = lambda prompt, fallback, must_contain=(): ("Priya is a young female candidate and a great fit.", "simulated-model")
try:
    demo_pool = PipelineCoordinator(log_path=pathlib.Path(tempfile.mkdtemp()) / "demo_log.txt").run()
finally:
    llm_module.generate_text = original

for item in demo_pool.bias_report[:2]:
    print(item["cid"], "flags:", item["flags"])
    print("  before:", item["original"])
    print("  after :", item["replaced_with"])

resume_01 flags: ['gender:female', 'age:young']
  before: Priya is a young female candidate and a great fit.
  after : Priya Sharma: 92/100. Meets 4 of 4 required skills; 4 yrs experience (min 3); education B.Tech Computer Science.
resume_02 flags: ['gender:female', 'age:young']
  before: Priya is a young female candidate and a great fit.
  after : Rahul Kumar: 76/100. Meets 3 of 4 required skills (missing: REST APIs); 3 yrs experience (min 3); education B.Sc Information Technology.


## 7. Model failure falls back to the template

In [8]:
dead = {"name": "dead-provider", "api_key": "x", "base_url": "http://127.0.0.1:9/v1", "model": "none"}
real = llm_module.active_providers
llm_module.active_providers = lambda: [dead]
try:
    text, source = llm_module.generate_text("Say hi", fallback="TEMPLATE SENTENCE", must_contain=("hi",))
finally:
    llm_module.active_providers = real
print("source:", source, "| text:", text)

source: template | text: TEMPLATE SENTENCE


## 8. The append-only log

In [9]:
log_lines = pathlib.Path(pool.log.path).read_text().splitlines()
print(f"{len(log_lines)} lines in logs/run_log.txt (grows on every run, never rewritten)")
print("\n".join(log_lines[-6:]))

344 lines in logs/run_log.txt (grows on every run, never rewritten)
2026-11-14T22:27:27 | 9bdc9c96 | scheduler_agent | not_shortlisted | resume_03 Aman Tiwari (rank 5, score 64)
2026-11-14T22:27:27 | 9bdc9c96 | scheduler_agent | not_shortlisted | resume_07 Arjun Mehta (rank 6, score 63)
2026-11-14T22:27:27 | 9bdc9c96 | scheduler_agent | not_shortlisted | resume_05 Vikram Singh (rank 7, score 43.7)
2026-11-14T22:27:27 | 9bdc9c96 | coordinator | stage_finished | schedule in 0.0s
2026-11-14T22:27:27 | 9bdc9c96 | coordinator | tool_call | build_final_report()
2026-11-14T22:27:27 | 9bdc9c96 | coordinator | run_finished | 3 interviews scheduled
